In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [3]:
df = pd.read_csv('../data/raw/ks-projects-201612.csv', encoding="ISO-8859-1", on_bad_lines='skip', low_memory=False)
df.columns = df.columns.str.strip() # Избавляемся от пробелов в названияхи

In [4]:
print(f"Cъехавших строк: {len(df[df['Unnamed: 13'].notna()])}") # 625

Cъехавших строк: 625


In [5]:
df = df.set_index("ID")
df = df.drop(["pledged", "backers", "usd pledged", "name", "Unnamed: 13", "Unnamed: 14", "Unnamed: 15", "Unnamed: 16"], axis=1)
df = df[df["state"].isin(["failed", "successful"])]

In [6]:
df['deadline'] = pd.to_datetime(df['deadline'])
df['launched'] = pd.to_datetime(df['launched'])

Добавляем новый признак: продолжительность компании

In [7]:
duration = df["deadline"] - df["launched"]
df["duration"] = duration.dt.days

# Признаки deadline и launched нам больше не нужны
df = df.drop(["deadline", "launched"], axis=1)

# ФИЛЬТРАЦИЯ АНОМАЛИЙ: оставляем только проекты длительностью от 1 до 90 дней
df = df[(df['duration'] > 0) & (df['duration'] <= 90)]

print(f"Осталось проектов: {len(df)}")

Осталось проектов: 280851


Label encoding

In [8]:
df["state"] = df["state"].map({"failed": 0, "successful": 1})

Train test split

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=["state"]), df["state"], test_size=0.2, random_state=42, stratify=df["state"])

Save data for Naive Bayes and Decision Tree

In [10]:
X_train, X_test, y_train, y_test

(                    category main_category currency    goal country  duration
 ID                                                                           
 1983703917            Shorts  Film & Video      USD    5000      US        30
 432964446            Fiction    Publishing      USD    1500      US        30
 215488935             Poetry    Publishing      USD    7968      US        29
 1418452386      Performances         Dance      USD    1800      US        13
 116697873             Crafts        Crafts      GBP   25000      GB        30
 ...                      ...           ...      ...     ...     ...       ...
 751285342            Fiction    Publishing      USD    1500      US        29
 909278702         Television  Film & Video      USD  370000      US        51
 1100185808  Children's Books    Publishing      USD    5000      US        29
 1697424416    Product Design        Design      USD     800      US        30
 1442392254        Technology    Technology      USD

In [11]:
X_train.to_csv('data/preprocessed/X_train.csv')
X_test.to_csv('data/preprocessed/X_test.csv')
y_train.to_csv('data/preprocessed/y_train.csv')
y_test.to_csv('data/preprocessed/y_test.csv')

Count Encoding for categorial features

In [12]:
for feature in ["category", "main_category", "currency", "country"]:
    counts = X_train[feature].value_counts()
    X_train[feature] = X_train[feature].map(counts)
    # Применяем к test и заполняем новые категории, которых не было в train, единицей
    X_test[feature] = X_test[feature].map(counts).fillna(1)

Normalization MinMax

In [13]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)

X_max = X_train.max()
X_min = X_train.min()

X_train = (X_train - X_min) / (X_max - X_min)
X_test = (X_test - X_min) / (X_max - X_min)

Save encoded data for K-NN

In [14]:
X_train, X_test, y_train, y_test

(            category  main_category  currency      goal   country  duration
 ID                                                                         
 1983703917  0.743148       1.000000  1.000000  0.000050  1.000000  0.325843
 432964446   0.495604       0.561863  1.000000  0.000015  1.000000  0.325843
 215488935   0.077487       0.561863  1.000000  0.000080  1.000000  0.314607
 1418452386  0.050336       0.000000  1.000000  0.000018  1.000000  0.134831
 116697873   0.234787       0.064346  0.103671  0.000250  0.103641  0.325843
 ...              ...            ...       ...       ...       ...       ...
 751285342   0.495604       0.561863  1.000000  0.000015  1.000000  0.314607
 909278702   0.047233       1.000000  1.000000  0.003700  1.000000  0.561798
 1100185808  0.346923       0.561863  1.000000  0.000050  1.000000  0.314607
 1697424416  1.000000       0.350896  1.000000  0.000008  1.000000  0.325843
 1442392254  0.325461       0.381475  1.000000  0.000250  1.000000  0.393258

In [15]:
X_train.to_csv('data/preprocessed/X_train_enc.csv')
X_test.to_csv('data/preprocessed/X_test_enc.csv')
y_train.to_csv('data/preprocessed/y_train_enc.csv')
y_test.to_csv('data/preprocessed/y_test_enc.csv')